In [1]:
import sys
from pathlib import Path

print("Python :", sys.version.split()[0])
print("Program:", sys.executable)
print("Folder :", Path.cwd())

_missing = []
for _name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

if _missing:
    print("STOP. Missing imports:", ", ".join(_missing))
else:
    print("All good. Continue.")


Python : 3.11.9
Program: D:\MLOps\.venv\Scripts\python.exe
Folder : C:\Users\vast\Downloads\P02-first-model
All good. Continue.


In [3]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error

DATA = Path("..") / "data" / "delivery_times.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} was not found. Complete P01/P02 setup or restore the shared course data."
    )

orders = pd.read_csv(DATA)

FEATURES = [
    "distance_km",
    "prep_time_min",
    "traffic_level",
    "rain",
]
TARGET = "delivery_min"

X = orders[FEATURES]
y = orders[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("dataset :", orders.shape)
print("develop :", X_train.shape, y_train.shape)
print("final   :", X_test.shape, y_test.shape)


dataset : (600, 5)
develop : (480, 4) (480,)
final   : (120, 4) (120,)


In [5]:
def score_both_ways(model, name):
    model.fit(X_train, y_train)

    train_mae = mean_absolute_error(
        y_train,
        model.predict(X_train),
    )
    test_mae = mean_absolute_error(
        y_test,
        model.predict(X_test),
    )
    gap = test_mae - train_mae

    result = {
        "model": name,
        "train": train_mae,
        "test": test_mae,
        "gap": gap,
    }

    print(
        f"{name:<28} "
        f"train {train_mae:5.2f}  "
        f"test {test_mae:5.2f}  "
        f"gap {gap:5.2f}"
    )

    return result


In [6]:
linear = score_both_ways(
    LinearRegression(),
    "LinearRegression",
)


LinearRegression             train  2.04  test  1.92  gap -0.11


In [7]:
tree_full = score_both_ways(
    DecisionTreeRegressor(random_state=42),
    "DecisionTree (unrestricted)",
)


DecisionTree (unrestricted)  train  0.00  test  3.43  gap  3.43


In [8]:
tree4 = score_both_ways(
    DecisionTreeRegressor(
        max_depth=4,
        random_state=42,
    ),
    "DecisionTree (depth 4)",
)


DecisionTree (depth 4)       train  3.66  test  4.23  gap  0.57


In [9]:
forest = score_both_ways(
    RandomForestRegressor(
        n_estimators=50,
        random_state=42,
    ),
    "RandomForest (50 trees)",
)


RandomForest (50 trees)      train  1.00  test  2.39  gap  1.39


In [10]:
diagnostic_table = pd.DataFrame(
    [linear, tree_full, tree4, forest]
).sort_values("test")

diagnostic_table.round(3)


,model,train,test,gap
0,LinearRegression,2.036,1.925,-0.112
3,RandomForest (50 trees),0.998,2.392,1.394
1,DecisionTree (unrestricted),0.000,3.426,3.426
2,DecisionTree (depth 4),3.658,4.228,0.570


In [11]:
def cross_validate_model(model, name):
    scores = -cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
    )

    print(
        f"{name:<28} "
        f"MAE {scores.mean():5.2f} "
        f"(+/- {scores.std():4.2f})"
    )

    return {
        "mean": float(scores.mean()),
        "std": float(scores.std()),
        "folds": scores,
    }


cv_linear = cross_validate_model(
    LinearRegression(),
    "LinearRegression",
)

cv_tree4 = cross_validate_model(
    DecisionTreeRegressor(
        max_depth=4,
        random_state=42,
    ),
    "DecisionTree (depth 4)",
)

cv_forest = cross_validate_model(
    RandomForestRegressor(
        n_estimators=50,
        random_state=42,
    ),
    "RandomForest (50 trees)",
)


LinearRegression             MAE  2.05 (+/- 0.17)
DecisionTree (depth 4)       MAE  4.44 (+/- 0.33)
RandomForest (50 trees)      MAE  2.75 (+/- 0.26)


In [12]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}

for depth in depths:
    candidate = DecisionTreeRegressor(
        max_depth=depth,
        random_state=42,
    )

    fold_mae = -cross_val_score(
        candidate,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
    )

    scores_by_depth[depth] = float(fold_mae.mean())

    print(
        f"max_depth={str(depth):<5} "
        f"MAE {fold_mae.mean():5.2f} "
        f"(+/- {fold_mae.std():4.2f})"
    )

best_depth = min(
    scores_by_depth,
    key=scores_by_depth.get,
)

print()
print("best development-CV depth:", best_depth)


max_depth=2     MAE  5.89 (+/- 0.44)
max_depth=3     MAE  4.81 (+/- 0.35)
max_depth=4     MAE  4.44 (+/- 0.33)
max_depth=6     MAE  3.65 (+/- 0.41)
max_depth=8     MAE  3.48 (+/- 0.27)
max_depth=None  MAE  3.51 (+/- 0.32)

best development-CV depth: 8


In [13]:
selected_name = "LinearRegression"
selected_model = LinearRegression()

selected_model.fit(X_train, y_train)

final_predictions = selected_model.predict(X_test)
final_test_mae = mean_absolute_error(
    y_test,
    final_predictions,
)

print("selected model :", selected_name)
print(f"final test MAE : {final_test_mae:.2f} min")


selected model : LinearRegression
final test MAE : 1.92 min


In [15]:
deep_tree = DecisionTreeRegressor(
    max_depth=12,
    random_state=42,
)

# fit deep_tree on X_train, y_train
deep_tree.fit(X_train, y_train)

# calculate the two MAEs
T1_train_mae = mean_absolute_error(
    y_train,
    deep_tree.predict(X_train)
)

T1_test_mae = mean_absolute_error(
    y_test,
    deep_tree.predict(X_test)
)

# test MAE minus training MAE
T1_gap = T1_test_mae - T1_train_mae

print(
    "train:", T1_train_mae,
    "test:", T1_test_mae,
    "gap:", T1_gap,
)

train: 0.04695601851851851 test: 3.397986111111111 gap: 3.3510300925925924


In [16]:
T2_depths = [2, 4, 6, None]

# Calculate mean positive CV MAE for each depth
T2_scores = {}

for depth in T2_depths:
    forest = RandomForestRegressor(
        n_estimators=50,
        max_depth=depth,
        random_state=42,
    )

    cv_scores = -cross_val_score(
        forest,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
    )

    T2_scores[depth] = cv_scores.mean()

# Choose the depth with the lowest MAE
T2_best_depth = min(T2_scores, key=T2_scores.get)

print(T2_scores)
print("best:", T2_best_depth)

{2: np.float64(5.422545124201891), 4: np.float64(3.5515271170143676), 6: np.float64(2.830879533681162), None: np.float64(2.7500833333333317)}
best: None


In [17]:
def compare(models):
    results = {}

    for name, model in models.items():
        scores = -cross_val_score(
            model,
            X_train,
            y_train,
            cv=5,
            scoring="neg_mean_absolute_error",
        )

        results[name] = {
            "cv_mae_mean": scores.mean(),
            "cv_mae_std": scores.std(),
        }

    return results


T3_table = compare({
    "linear": LinearRegression(),
    "tree4": DecisionTreeRegressor(
        max_depth=4,
        random_state=42,
    ),
    "forest": RandomForestRegressor(
        n_estimators=50,
        random_state=42,
    ),
})

print(T3_table)

{'linear': {'cv_mae_mean': np.float64(2.0518478822261157), 'cv_mae_std': np.float64(0.16802410875985976)}, 'tree4': {'cv_mae_mean': np.float64(4.437483819525063), 'cv_mae_std': np.float64(0.3298979287536765)}, 'forest': {'cv_mae_mean': np.float64(2.7500833333333317), 'cv_mae_std': np.float64(0.25501859375293084)}}


In [18]:
# ------------------------------------------------------------------
# SELF-CHECK
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check(
    "T1 | depth-12 tree is trained",
    lambda: hasattr(deep_tree, "tree_"),
)

_check(
    "T1 | train and test MAE match deep_tree",
    lambda:
        abs(
            float(T1_train_mae)
            - mean_absolute_error(
                y_train,
                deep_tree.predict(X_train),
            )
        ) < 0.01
        and
        abs(
            float(T1_test_mae)
            - mean_absolute_error(
                y_test,
                deep_tree.predict(X_test),
            )
        ) < 0.01,
)

_check(
    "T1 | gap is test minus train and positive",
    lambda:
        abs(
            float(T1_gap)
            - (
                float(T1_test_mae)
                - float(T1_train_mae)
            )
        ) < 1e-6
        and float(T1_gap) > 0,
)

_check(
    "T2 | one forest-CV score exists per requested depth",
    lambda: set(T2_scores) == {2, 4, 6, None},
)

_check(
    "T2 | scores match development-only 5-fold CV",
    lambda: all(
        abs(
            float(T2_scores[d])
            - float(
                (
                    -cross_val_score(
                        RandomForestRegressor(
                            n_estimators=50,
                            max_depth=d,
                            random_state=42,
                        ),
                        X_train,
                        y_train,
                        cv=5,
                        scoring="neg_mean_absolute_error",
                    )
                ).mean()
            )
        ) < 0.01
        for d in [2, 4, 6, None]
    ),
)

_check(
    "T2 | best depth is the lowest-CV-MAE depth",
    lambda:
        T2_best_depth
        == min(T2_scores, key=T2_scores.get),
)

_check(
    "T3 | compare returns the requested model names and metrics",
    lambda:
        set(T3_table) == {"linear", "tree4", "forest"}
        and all(
            set(T3_table[name])
            == {"cv_mae_mean", "cv_mae_std"}
            for name in T3_table
        ),
)

_linear_cv_reference = -cross_val_score(
    LinearRegression(),
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
)

_check(
    "T3 | linear CV mean and std match development-only CV",
    lambda:
        abs(
            float(T3_table["linear"]["cv_mae_mean"])
            - float(_linear_cv_reference.mean())
        ) < 0.01
        and
        abs(
            float(T3_table["linear"]["cv_mae_std"])
            - float(_linear_cv_reference.std())
        ) < 0.01,
)

print("=" * 72)
print("SELF-CHECK   Practical 03: Choosing a Model Honestly")
print("=" * 72)

for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")

print("-" * 72)

_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("=" * 72)

if _passed == len(_results):
    print("Well done. Add the model-choice justification and save the notebook.")
else:
    print("Read the FAIL lines, fix those tasks, and run this cell again.")


SELF-CHECK   Practical 03: Choosing a Model Honestly
  [PASS]  T1 | depth-12 tree is trained
  [PASS]  T1 | train and test MAE match deep_tree
  [PASS]  T1 | gap is test minus train and positive
  [PASS]  T2 | one forest-CV score exists per requested depth
  [PASS]  T2 | scores match development-only 5-fold CV
  [PASS]  T2 | best depth is the lowest-CV-MAE depth
  [PASS]  T3 | compare returns the requested model names and metrics
  [PASS]  T3 | linear CV mean and std match development-only CV
------------------------------------------------------------------------
  8 of 8 checks passed
Well done. Add the model-choice justification and save the notebook.
